# Danelfin File Processing
[Danelfin](https://danelfin.com/how-it-works#ai-score) provides a stock analytics platform powered by Explainable Artificial Intelligence. It rates stocks and ETFs with an easy-to-understand global AI Score, ranging from 1 to 10; the higher the score, the higher the probability of beating the market in the next 3 months.

-   **dfin_top100.json** -- Top 100 ranking of AI Scores for the specified date.
-   **dfin_trade_ideas.json** -- A filterable, paginated list of 25 most recent Trade Ideas: stocks and ETFs with an active long or short signal where Danelfin's AI has historically performed best, , sorted by 3-month win rate in descending order. 
-   **{symbol}.ai_scores.json** -- AI Score rates stocks and ETFs according to the probability of beating the market in the next 3 months. Stocks and ETFs with the highest AI Scores (7 to 10) have a higher probability of outperforming the market, while stocks and ETFs with the lowest AI Scores (1 to 3) have a lower probability.
-   **{symbol}.price_forecast.json** -- Danelfin's price forecast model output for a single stock across four horizons (1m, 3m, 6m, 1y): median expected return and 68%/90% prediction bands, plus take-profit and stop-loss thresholds for the 3-month horizon.
-   **{symbol}.trading_params.json** -- Entry price, stop-loss, and take-profit levels for a single stock or ETF, as absolute prices and as percentages of the entry price, plus the signal, investment horizon, and currency.

This notebook exercises the **DanelfinClient** (danelfin.py) for interactive experimentation.

ref:
[Danelfin API Documentation](https://danelfin.com/docs/api)



In [ ]:
# local ...
import sys, os
sys.path.append("./backend/danelfin")
print(sys.path)

os.environ["INGEST_PATH"] = "/home/richa/projects/scratchpad/ingest/market_data"

In [ ]:
#ToDo: Query dynamoDB 'postions' table for symbols instead of hardcoding them
symbols = """FSTA
GLD
HODL
IDHQ
IVW
JNJ
LVHI
QQQ
RYCEY
SLV
SPY
VYM
WMT
XLK""".splitlines()

symbols

In [ ]:
""" danelfin_fetch.py: Lambda function to fetch Danelfin rankings into .json files """

# local
from danelfin_fetch import lambda_handler

# mimic event created by EventBridge scheduled rule
event = {
    "symbols": ["FSTA", "GLD", "HODL", "IDHQ", "IVW", "JNJ", "LVHI", "QQQ", "SLV", "SPY", "VYM", "WMT", "XLK"],
    "today_str": None
}

lambda_handler(event, context=None)



In [1]:
import danelfin 
from datetime import datetime

client = danelfin.Client()
today_str = event["today_str"]
if not today_str:
    today_str = datetime.today().strftime("%Y-%m-%d")

# 1. Historical data for a ticker
pypl_history = client.ranking(ticker="XLK")
pypl_history


ModuleNotFoundError: No module named 'requests'

In [ ]:
# 2. Top 100 tickers for a specific day
top_100 = client.ranking(date=today_str)
#top_100_tickers = [entry['ticker'] for entry in top_100]
top_100


In [ ]:
top_100_tickers = [entry['ticker'] for entry in top_100]
print(top_100_tickers)

In [ ]:
# read the top 100 tickers from the file

import json

with open("2026-05-28.TOP_100.json", "r") as f:
    top_100_tickers_from_file = json.load(f)
print(top_100_tickers_from_file)


In [ ]:
# 3. Ticker data for a specific day
pypl_day = client.ranking(date=today_str, ticker="GLD")


In [ ]:
# 4. Filter by AI Score
aiscore_10 = client.ranking(date=today_str, aiscore=10)


In [ ]:

# 5. Buy track record
buy_track = client.ranking(date=today_str, buy_track_record=1)


In [ ]:

# 6. Sector filter
utilities = client.ranking(date=today_str, sector="utilities")


In [ ]:

# 7. Sector + low risk filter
utilities_lowrisk = client.ranking(
    date=today_str,
    sector="utilities",
    low_risk_min=5
)


In [ ]:

# 8. Industry filter
banks = client.ranking(date=today_str, industry="banks")


In [ ]:

# 9. Custom fields
custom = client.ranking(
    date=today_str,
    fields="date,ticker,signal,aiscore"
)

# 10. European market
sap = client.ranking(ticker="SAP.DE", market="europe")


In [ ]:
# 11. Trade ideas
trade_ideas = client.trade_ideas()
trade_ideas

In [ ]:
# 12. Trading parameters
trading_parameters = client.trading_parameters(ticker="GLD")


In [ ]:
sectors = client.list_sectors()
print(json.dumps(sectors, indent=4))

In [ ]:
sector_scores = client.sector_scores(slug="energy")
print(json.dumps(sector_scores, indent=4))

In [ ]:
industries = client.list_industries()
print(json.dumps(industries, indent=4))

In [ ]:
price_forecast = client.price_forecast(ticker="GLD")
print(json.dumps(price_forecast, indent=4))